# Phase 3-4 调试 Notebook — 3D ResNet + Multi-Backbone Ensemble

> 在 Kaggle 环境或有真实 DICOM 数据的机器上逐 cell 运行，验证 3D 和 Ensemble 管线。
>
> 本地无 DICOM 时，模型结构和 shape 验证可完整测试。

## Phase 3 检查清单
| # | 检查项 | 数据依赖 | 预期 |
|---|--------|----------|------|
| 1 | 环境 & 导入 | 无 | 无报错，ResNet3D/ConvNeXt/Swin 可导入 |
| 2 | 配置加载 (phase3) | 无 | YAML 解析成功 |
| 3 | VolumeDataset 构建 | DICOM | 样本数 > 0, volume [1,32,128,128] |
| 4 | ResNet3D Model forward | 无 | [B,1,32,128,128] → [B,12] |
| 5 | Gradient Checkpointing 验证 | 无 | 训练模式下显存节省 (VRAM 对比) |
| 6 | 3D 过拟合测试 | DICOM | 单 batch loss→0 |
## Phase 4 检查清单
| # | 检查项 | 数据依赖 | 预期 |
|---|--------|----------|------|
| 7 | ConvNeXt25D forward | 无 | [B,5,384,384] → [B,12] |
| 8 | Swin25D forward | 无 | [B,5,384,384] → [B,12] |
| 9 | EnsembleModel forward | 无 | 多 backbone 融合 → [B,12] |
| 10 | EnsembleInference 推理 | 无 (mock ckpt) | 加权平均概率正确 |
| 11 | ensemble_submissions 合并 | 无 | 多 submission CSV 加权平均 |
| 12 | 完整 ensemble 干跑 | DICOM | backbone 训练 + 推理不报错 |

In [ ]:
# ============================================================
# Cell 1: 环境 & 导入
# ============================================================
from __future__ import annotations
import sys, time, json, tempfile
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yaml
import matplotlib.pyplot as plt

deps = {}
for name in ["torch", "numpy", "pandas", "yaml", "cv2", "pydicom", "timm", "sklearn"]:
    try:
        __import__(name)
        deps[name] = "OK"
    except ImportError:
        deps[name] = "MISSING"

print(f"Python {sys.version.split()[0]}  |  PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB")
print("  " + "  ".join(f"{k}:{v}" for k, v in deps.items()))

In [ ]:
# ============================================================
# Cell 2: 配置加载 — Phase 3 & Phase 4 configs
# ============================================================
import yaml

configs = {}
for name, path in [
    ("phase3", "configs/phase3_3d.yaml"),
    ("phase4", "configs/phase4_ensemble.yaml"),
]:
    with open(PROJECT_ROOT / path, encoding="utf-8") as f:
        configs[name] = yaml.safe_load(f)

# Phase 3 配置检查
c3 = configs["phase3"]
print("Phase 3 — 3D ResNet 配置:")
print(f"  Stage:       {c3['experiment']['stage']}")
print(f"  Model:       resnet3d_18 (in_ch={c3['model']['in_channels']}, pretrained={c3['model']['pretrained']})")
print(f"  Grad ckpt:   {c3['model']['use_grad_checkpoint']}")
print(f"  Volume:      {c3['data']['volume_depth']}×{c3['data']['volume_size']}×{c3['data']['volume_size']}")
print(f"  Batch:       {c3['train']['batch_size']} × accum {c3['train']['gradient_accumulation_steps']} = effective {c3['train']['batch_size']*c3['train']['gradient_accumulation_steps']}")
print(f"  AMP:         {c3['train']['mixed_precision']}")
print(f"  VRAM target: ~8GB (batch=1, depth=32, size=128, grad_ckpt+AMP)")

# Phase 4 配置检查
c4 = configs["phase4"]
print(f"\nPhase 4 — Ensemble Backbone 配置:")
print(f"  Stage:       {c4['experiment']['stage']}")
print(f"  Arch:        {c4['model']['arch']}")
print(f"  Planes:      {c4['data']['planes']}")
print(f"  Dropout:     {c4['model']['dropout']}")
print(f"  Pseudo:      enabled={c4['pseudo_label']['enabled']}, conf={c4['pseudo_label']['confidence']}")

# Ensemble 阵容
ens = c4.get("ensemble", {})
print(f"\n  Ensemble 阵容:")
for m in ens.get("models", []):
    print(f"    {m['name']:<30s} weight={m['weight']:.2f}")

In [ ]:
# ============================================================
# Cell 3: VolumeDataset 构建 & 样本检查
# ============================================================
# 注意: 此 cell 仅在 DICOM 文件存在时产生样本

from datasets import VolumeDataset, PseudoLabelLoader
from utils import TARGET_COLUMNS

c3 = configs["phase3"]

# 尝试加载标签
series_csv = c3["paths"].get("series_csv", "data/metadata/train_series.csv")
series_path = PROJECT_ROOT / series_csv

if series_path.exists():
    series_df = pd.read_csv(series_path)
    print(f"Series 元数据: {len(series_df):,} rows, {series_df['StudyInstanceUID'].nunique():,} studies")
    
    # Sagittal 覆盖率
    sag_series = series_df[series_df["Anatomical_Plane"] == "Sagittal"]
    sag_studies = sag_series["StudyInstanceUID"].nunique()
    print(f"  Sagittal series: {len(sag_series):,}, studies: {sag_studies:,}")
else:
    print(f"⚠️  Series CSV 不存在: {series_path}")
    series_df = None

# 尝试构建 VolumeDataset
if series_df is not None:
    # 最小标签 — 用 series 中的 study uid 构建 dummy labels
    sag_study_ids = sorted(series_df[series_df["Anatomical_Plane"] == "Sagittal"]["StudyInstanceUID"].unique())
    dummy_labels = pd.DataFrame({
        "StudyInstanceUID": sag_study_ids,
        **{col: 0 for col in TARGET_COLUMNS},
    }).set_index("StudyInstanceUID")
    
    ds = VolumeDataset(
        series_df=series_df,
        labels_df=dummy_labels,
        dicom_root=c3["paths"]["dicom_root"],
        volume_depth=c3["data"]["volume_depth"],
        volume_size=c3["data"]["volume_size"],
        plane=c3["data"].get("plane", "Sagittal"),
        is_train=True,
    )
    
    print(f"\nVolumeDataset: {len(ds)} volumes")
    
    if len(ds) > 0:
        sample = ds[0]
        print(f"\n  Sample [0]:")
        print(f"    volume:    {list(sample['volume'].shape)} ({sample['volume'].dtype})")
        print(f"    labels:    {list(sample['labels'].shape)}")
        print(f"    study_uid: {sample['study_uid'][:50]}...")
        print(f"    range:     [{sample['volume'].min():.3f}, {sample['volume'].max():.3f}]")
        print(f"    mean/std:  {sample['volume'].mean():.3f} / {sample['volume'].std():.3f}")
        
        # 检查是否存在全零 volume (DICOM 读取失败)
        is_zero = (sample['volume'].abs().sum() == 0).item()
        print(f"    all-zero:  {is_zero} {'⚠️ DICOM 读取失败' if is_zero else 'OK'}")
        
        # 可视化中间切片
        mid_slice = sample['volume'][0, 16].numpy()  # depth=32, 取中间
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(mid_slice, cmap="gray")
        ax.set_title(f"Volume slice 16/32 — {sample['study_uid'][:40]}...")
        ax.axis("off")
        plt.tight_layout()
        plt.show()
    else:
        print("  ⚠️  VolumeDataset 为空 — DICOM 数据不可用")
else:
    print("⚠️  跳过 VolumeDataset 测试 — 无 series 元数据")

In [ ]:
# ============================================================
# Cell 4: ResNet3DModel Forward Pass 验证
# ============================================================
from models import ResNet3DModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

c3 = configs["phase3"]
mc = c3["model"]

# ── 测试 1: 标准 forward (inference mode) ──────────────────
print("\n--- 测试 1: Inference mode forward ---")
model = ResNet3DModel(
    in_channels=mc["in_channels"],
    num_classes=mc["num_classes"],
    pretrained=False,  # 快速测试
    dropout=mc["dropout"],
    use_grad_checkpoint=mc.get("use_grad_checkpoint", True),
).to(device)
model.eval()

n_p = sum(p.numel() for p in model.parameters()) / 1e6
n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"  Params: {n_p:.1f}M total, {n_tr:.1f}M trainable")

# Forward test — 各种 batch size
for bs in [1, 2]:
    x = torch.randn(bs, 1, 32, 128, 128).to(device)
    with torch.no_grad():
        out = model(x)
    print(f"  batch={bs}: input {list(x.shape)} → output {list(out.shape)}")
    assert out.shape == (bs, 12), f"Shape mismatch: {out.shape}"

print("  OK: Inference mode shapes correct")

# ── 测试 2: Training mode (grad checkpoint) ─────────────────
print("\n--- 测试 2: Training mode (gradient checkpointing) ---")
model.train()

x = torch.randn(1, 1, 32, 128, 128).to(device)
out = model(x)
loss = out.sum()
loss.backward()

# 检查梯度是否正常传播
grad_norms = {}
for name, param in model.named_parameters():
    if param.grad is not None:
        grad_norms[name] = param.grad.norm().item()

if grad_norms:
    max_grad_name = max(grad_norms, key=grad_norms.get)
    min_grad_name = min(grad_norms, key=grad_norms.get)
    print(f"  Gradients flowing: {len(grad_norms)} params with grad")
    print(f"    Max grad norm: {grad_norms[max_grad_name]:.4f} ({max_grad_name[:60]}...)")
    print(f"    Min grad norm: {grad_norms[min_grad_name]:.4f} ({min_grad_name[:60]}...)")
    print(f"  OK: Gradient checkpointing works — backward pass successful")
else:
    print(f"  WARN: No gradients — check model setup")

# ── 测试 3: extract_features ────────────────────────────────
print("\n--- 测试 3: extract_features (for ensemble) ---")
model.eval()
with torch.no_grad():
    feats = model.extract_features(x)
print(f"  extract_features: input {list(x.shape)} → features {list(feats.shape)}")
print(f"  Expected feature_dim: {model.feature_dim}, actual: {feats.shape[1]}")
assert feats.shape == (1, model.feature_dim), f"Feature shape mismatch!"
print(f"  OK: extract_features shape correct")

del model, x, out, feats

In [ ]:
# ============================================================
# Cell 5: Gradient Checkpointing VRAM 对比
# ============================================================
if device == "cuda":
    print("Gradient Checkpointing VRAM 对比:\n")
    
    x = torch.randn(1, 1, 32, 128, 128).to(device)
    target = torch.randn(1, 12).to(device)
    
    # ── Without gradient checkpointing ──────────────────────
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    model_no_ckpt = ResNet3DModel(
        in_channels=1, num_classes=12, pretrained=False,
        use_grad_checkpoint=False,
    ).to(device)
    model_no_ckpt.train()
    
    out = model_no_ckpt(x)
    loss = nn.BCEWithLogitsLoss()(out, target)
    loss.backward()
    
    vram_no_ckpt = torch.cuda.max_memory_allocated() / 1024**3
    
    del model_no_ckpt, out, loss
    torch.cuda.empty_cache()
    
    # ── With gradient checkpointing ─────────────────────────
    torch.cuda.reset_peak_memory_stats()
    
    model_ckpt = ResNet3DModel(
        in_channels=1, num_classes=12, pretrained=False,
        use_grad_checkpoint=True,
    ).to(device)
    model_ckpt.train()
    
    out = model_ckpt(x)
    loss = nn.BCEWithLogitsLoss()(out, target)
    loss.backward()
    
    vram_ckpt = torch.cuda.max_memory_allocated() / 1024**3
    
    print(f"  Without grad checkpoint:  {vram_no_ckpt:.2f} GB")
    print(f"  With grad checkpoint:     {vram_ckpt:.2f} GB")
    print(f"  VRAM saved:               {vram_no_ckpt - vram_ckpt:.2f} GB ({(1 - vram_ckpt/max(vram_no_ckpt, 0.001))*100:.0f}%)")
    print(f"  {'OK: Gradient checkpointing 有效节省 VRAM' if vram_ckpt < vram_no_ckpt else 'Note: 单 batch 差异不明显，大 batch 下更显著'}")
    
    del model_ckpt, x, target
    torch.cuda.empty_cache()
else:
    print("⚠️  无 CUDA, 跳过 VRAM 对比。Gradient checkpointing 在模型定义中已配置。")

In [ ]:
# ============================================================
# Cell 6: 3D 过拟合测试
# ============================================================
from losses import FocalBCELoss
from utils import compute_macro_auc

# 查看 VolumeDataset 是否有数据
try:
    ds_len = len(ds)
except NameError:
    ds_len = 0

if ds_len > 0:
    print("3D Overfitting Test")
    
    # 取 1 个 volume 做单样本过拟合
    sample = ds[0]
    x = sample["volume"].unsqueeze(0).to(device)  # [1, 1, 32, 128, 128]
    y = sample["labels"].unsqueeze(0).to(device)   # [1, 12]
    
    print(f"  Overfitting on 1 volume: {sample['study_uid'][:50]}...")
    print(f"  Labels: {[TARGET_COLUMNS[i] for i, v in enumerate(y[0].cpu().numpy()) if v > 0] or 'all negative'}")
    
    test_model = ResNet3DModel(
        in_channels=1, num_classes=12, pretrained=False,
        use_grad_checkpoint=False,  # 过拟合测试不需要
    ).to(device)
    
    criterion = FocalBCELoss(gamma=2.0, alpha=0.25)
    optimizer = torch.optim.AdamW(test_model.parameters(), lr=1e-3)
    
    loss_history = []
    MAX_STEPS = 300
    
    for step in range(MAX_STEPS):
        test_model.train()
        optimizer.zero_grad()
        out = test_model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        loss_history.append(loss.item())
        
        if step % 60 == 0 or loss.item() < 0.001:
            with torch.no_grad():
                auc = compute_macro_auc(y.cpu().numpy(), out.detach().cpu().numpy())
            print(f"  step {step:4d}: loss={loss.item():.6f}  AUC={auc:.4f}")
        
        if loss.item() < 0.001:
            print(f"  3D 过拟合成功! (step {step})")
            break
    else:
        print(f"  {MAX_STEPS} steps 完成, final loss={loss_history[-1]:.6f}")
        if loss_history[-1] > 0.01:
            print(f"  WARN: 过拟合可能不充分")
    
    # Loss 曲线
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(loss_history)
    ax.set_xlabel("Step"); ax.set_ylabel("Loss")
    ax.set_title("3D ResNet Overfitting Test — Single Volume")
    ax.set_yscale("log"); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
    
    del test_model, x, y
else:
    print("⚠️  无 DICOM 数据，跳过 3D 过拟合测试。")
    print("   在 Kaggle 上运行时此 cell 将验证 ResNet3D 能否在单 volume 上收敛。")

In [ ]:
# ============================================================
# Cell 7: ConvNeXt25D Forward Pass 验证
# ============================================================
from models import ConvNeXt25D

print("--- ConvNeXt25D Forward Pass ---")

model = ConvNeXt25D(
    in_channels=5, num_classes=12, pretrained=False, dropout=0.3,
).to(device)

n_p = sum(p.numel() for p in model.parameters()) / 1e6
n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"  Params: {n_p:.1f}M total, {n_tr:.1f}M trainable")
print(f"  Actual feature_dim: {model.feature_dim}")

for bs in [1, 2, 4, 8]:
    x = torch.randn(bs, 5, 384, 384).to(device)
    with torch.no_grad():
        out = model(x)
        feats = model.extract_features(x)
    print(f"  batch={bs}: input {list(x.shape)} → logits {list(out.shape)}, features {list(feats.shape)}")
    assert out.shape == (bs, 12), f"Shape mismatch!"
    assert feats.shape == (bs, model.feature_dim), f"Feature shape mismatch!"

# 检查梯度
model.train()
x = torch.randn(2, 5, 384, 384).to(device)
out = model(x)
loss = out.sum()
loss.backward()

grad_count = sum(1 for p in model.parameters() if p.grad is not None)
print(f"\n  Gradients: {grad_count} params with grad (OK)")

del model, x, out, loss
print("  ConvNeXt25D: All tests passed")
torch.cuda.empty_cache() if device == "cuda" else None

In [ ]:
# ============================================================
# Cell 8: Swin25D Forward Pass 验证
# ============================================================
from models import Swin25D

print("--- Swin25D Forward Pass ---")

model = Swin25D(
    in_channels=5, num_classes=12, pretrained=False, dropout=0.3, window_size=7,
).to(device)

n_p = sum(p.numel() for p in model.parameters()) / 1e6
n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"  Params: {n_p:.1f}M total, {n_tr:.1f}M trainable")
print(f"  Actual feature_dim: {model.feature_dim}")

# 测试不同尺寸 (Swin 对 img_size 有要求)
for size in [224, 384]:
    for bs in [1, 2]:
        x = torch.randn(bs, 5, size, size).to(device)
        with torch.no_grad():
            out = model(x)
            feats = model.extract_features(x)
        print(f"  {size}×{size}, batch={bs}: → logits {list(out.shape)}, features {list(feats.shape)}")
        assert out.shape == (bs, 12), f"Shape mismatch!"
        assert feats.shape[1] == model.feature_dim, f"Feature dim mismatch!"

# 检查 forward_features 输出格式
model.eval()
with torch.no_grad():
    raw_feats = model.backbone.forward_features(torch.randn(1, 5, 384, 384).to(device))
print(f"\n  forward_features output: dim={raw_feats.dim()}, shape={list(raw_feats.shape)}")
print(f"  {'OK: Swin handles both [B,N,D] and [B,D,H,W] formats' if raw_feats.dim() in [3, 4] else 'WARN: unexpected dim'}")

# 梯度测试
model.train()
x = torch.randn(1, 5, 384, 384).to(device)
out = model(x)
out.sum().backward()

grad_count = sum(1 for p in model.parameters() if p.grad is not None)
print(f"  Gradients: {grad_count} params with grad (OK)")

del model, x, out
print("  Swin25D: All tests passed")
torch.cuda.empty_cache() if device == "cuda" else None

In [ ]:
# ============================================================
# Cell 9: EnsembleModel Forward Pass
# ============================================================
from models import EfficientNetV2S25D, ConvNeXt25D, Swin25D
from models.ensemble import EnsembleModel

print("--- EnsembleModel 测试 ---")

# 构建 3 个 backbone (不用 pretrained — 快速测试)
effnet = EfficientNetV2S25D(in_channels=5, num_classes=12, pretrained=False, dropout=0.3).to(device)
convnext = ConvNeXt25D(in_channels=5, num_classes=12, pretrained=False, dropout=0.3).to(device)
swin = Swin25D(in_channels=5, num_classes=12, pretrained=False, dropout=0.3).to(device)

backbones = [effnet, convnext, swin]
feature_dims = [b.feature_dim for b in backbones]
print(f"  Backbones: EfficientNetV2-S ({feature_dims[0]}d), ConvNeXt-S ({feature_dims[1]}d), Swin-T ({feature_dims[2]}d)")

# ── 测试 1: Weighted Mean (learnable weights) ─────────────
print("\n--- Test 1: Weighted Mean Fusion ---")
ensemble_wm = EnsembleModel(
    backbones=[effnet, convnext, swin],
    feature_dims=feature_dims,
    num_classes=12,
    fusion="weighted_mean",
    learnable_weights=True,
    dropout=0.3,
).to(device)

n_p = sum(p.numel() for p in ensemble_wm.parameters()) / 1e6
print(f"  Total params: {n_p:.1f}M")
print(f"  Model weights (learnable): {ensemble_wm.model_weights.data.softmax(dim=0).tolist()}")

x = torch.randn(4, 5, 384, 384).to(device)
with torch.no_grad():
    out = ensemble_wm(x)
print(f"  Forward: {list(x.shape)} → {list(out.shape)}")
assert out.shape == (4, 12)

# 梯度测试
ensemble_wm.train()
out = ensemble_wm(x)
loss = out.sum()
loss.backward()

# 检查 model_weights 梯度
if ensemble_wm.model_weights.grad is not None:
    print(f"  model_weights grad: {ensemble_wm.model_weights.grad.tolist()}")
    print(f"  OK: Learnable weights receive gradients")

# ── 测试 2: Concat Fusion ──────────────────────────────────
print("\n--- Test 2: Concat Fusion ---")
ensemble_cat = EnsembleModel(
    backbones=[effnet, convnext, swin],
    feature_dims=feature_dims,
    num_classes=12,
    fusion="concat",
    dropout=0.3,
).to(device)

n_p2 = sum(p.numel() for p in ensemble_cat.parameters()) / 1e6
print(f"  Total params: {n_p2:.1f}M")
print(f"  Fusion: 3×features({sum(feature_dims)}d total) → Linear({sum(feature_dims)}, {max(feature_dims)}) → head")

with torch.no_grad():
    out = ensemble_cat(x)
print(f"  Forward: {list(x.shape)} → {list(out.shape)}")
assert out.shape == (4, 12)

print("\n  EnsembleModel: Both fusion modes work correctly")

del ensemble_wm, ensemble_cat, effnet, convnext, swin, x
torch.cuda.empty_cache() if device == "cuda" else None

In [ ]:
# ============================================================
# Cell 10: EnsembleInference — 推理时加权平均
# ============================================================
# 测试: 创建 3 个 mock checkpoint, 验证加权平均结果

from models import EfficientNetV2S25D, ConvNeXt25D, Swin25D
from models.ensemble import EnsembleInference

print("--- EnsembleInference 测试 ---")

# 创建临时 checkpoint
tmp_dir = Path(tempfile.mkdtemp(prefix="ensemble_test_"))

model_configs = [
    (EfficientNetV2S25D, {"in_channels": 5, "num_classes": 12, "pretrained": False, "dropout": 0.3}, "effnet_mock.pt"),
    (ConvNeXt25D, {"in_channels": 5, "num_classes": 12, "pretrained": False, "dropout": 0.3}, "convnext_mock.pt"),
    (Swin25D, {"in_channels": 5, "num_classes": 12, "pretrained": False, "dropout": 0.3}, "swin_mock.pt"),
]

ckpt_paths = []
model_classes = []
model_kwargs = []

for cls, kwargs, fname in model_configs:
    model = cls(**kwargs)
    path = tmp_dir / fname
    torch.save({"model": model.state_dict(), "auc": 0.75, "epoch": 10}, path)
    ckpt_paths.append(str(path))
    model_classes.append(cls)
    model_kwargs.append(kwargs)
    print(f"  Created mock ckpt: {fname}")

# ── 测试 1: 等权平均 ─────────────────────────────────────
print("\n--- Test 1: Equal weights ---")
ensemble_inf = EnsembleInference(
    checkpoint_paths=ckpt_paths,
    model_classes=model_classes,
    model_kwargs=model_kwargs,
    weights=None,  # 等权
    device=device,
)

x = torch.randn(4, 5, 384, 384).to(device)
probs = ensemble_inf.predict(x)
print(f"  Input: {list(x.shape)} → Probs: {list(probs.shape)}")
print(f"  Prob range: [{probs.min():.3f}, {probs.max():.3f}] (expected 0-1)")
assert probs.shape == (4, 12)
assert (probs >= 0).all() and (probs <= 1).all(), "Probs out of range!"
print(f"  OK: Equal-weight inference correct")

# ── 测试 2: 自定义权重 ─────────────────────────────────────
print("\n--- Test 2: Custom weights ---")
ensemble_inf2 = EnsembleInference(
    checkpoint_paths=ckpt_paths,
    model_classes=model_classes,
    model_kwargs=model_kwargs,
    weights=[0.5, 0.3, 0.2],
    device=device,
)

probs2 = ensemble_inf2.predict(x)
print(f"  Probs range: [{probs2.min():.3f}, {probs2.max():.3f}]")
assert probs2.shape == (4, 12)
print(f"  OK: Custom-weight inference correct")

# 权重不同 → 输出不同
diff = (probs - probs2).abs().mean().item()
print(f"  Equal vs custom weight diff: {diff:.4f} (should be > 0)")

del ensemble_inf, ensemble_inf2
print(f"\n  EnsembleInference: All tests passed")
torch.cuda.empty_cache() if device == "cuda" else None

In [ ]:
# ============================================================
# Cell 11: ensemble_submissions — 合并多个 CSV
# ============================================================
from models.ensemble import ensemble_submissions
from utils import TARGET_COLUMNS

print("--- ensemble_submissions 测试 ---")

# 创建 3 个 mock submission CSV
study_ids = [f"1.2.826.0.1.{i:010d}" for i in range(100, 200)]

np.random.seed(42)
mock_subs = []
for i in range(3):
    data = np.random.rand(len(study_ids), 12).astype(np.float32)
    # 给每个模型稍微不同的预测
    data += np.random.normal(0, 0.05, data.shape)
    data = np.clip(data, 0.01, 0.99)
    df = pd.DataFrame(data, index=study_ids, columns=TARGET_COLUMNS)
    df.index.name = "StudyInstanceUID"
    path = tmp_dir / f"submission_model_{i}.csv"
    df.to_csv(path)
    mock_subs.append(str(path))
    print(f"  Created mock submission: submission_model_{i}.csv ({len(df)} studies)")

# ── Test: 等权合并 ────────────────────────────────────────
print("\n--- Test: Equal-weight merge ---")
merged = ensemble_submissions(mock_subs, weights=None)
print(f"  Merged: {len(merged)} studies × {len(merged.columns)} classes")
print(f"  Index name: {merged.index.name}")
print(f"  Columns: {list(merged.columns[:4])}...")
print(f"  Range: [{merged.values.min():.3f}, {merged.values.max():.3f}]")

# ── Test: 加权合并 ────────────────────────────────────────
print("\n--- Test: Weighted merge ---")
merged_w = ensemble_submissions(mock_subs, weights=[0.5, 0.3, 0.2])
print(f"  Merged with weights [0.5, 0.3, 0.2]: {len(merged_w)} studies")

# 等权 vs 加权应有差异
diff = (merged.values - merged_w.values).abs().mean()
print(f"  Equal vs weighted diff: {diff:.4f} (should be > 0)")

# ── Test: 保存到文件 ──────────────────────────────────────
output_path = tmp_dir / "submission_ensemble_test.csv"
merged_saved = ensemble_submissions(mock_subs, weights=None, output_path=output_path)
print(f"\n  Saved to: {output_path}")
print(f"  File exists: {output_path.exists()}")
print(f"  File size: {output_path.stat().st_size} bytes")

print(f"\n  ensemble_submissions: All tests passed")

In [ ]:
# ============================================================
# Cell 12: 完整 Ensemble Backbone 干跑 (2 epochs)
# ============================================================
# 验证单个 backbone 的训练循环不报错

from torch.utils.data import DataLoader
from datasets import Knee25DDataset
from losses import FocalBCELoss
from utils import compute_macro_auc, format_per_class_auc

c4 = configs["phase4"]

# 检查 DICOM 数据
dicom_path = PROJECT_ROOT / c4["paths"]["dicom_root"]
series_path = PROJECT_ROOT / c4["paths"].get("series_csv", "data/metadata/train_series.csv")

if series_path.exists() and dicom_path.exists():
    series_df = pd.read_csv(series_path)
    sag_studies = sorted(series_df[series_df["Anatomical_Plane"] == "Sagittal"]["StudyInstanceUID"].unique())
    
    # Dummy labels
    dummy_labels = pd.DataFrame({
        "StudyInstanceUID": sag_studies[:20],
        **{col: np.random.randint(0, 2) for col in TARGET_COLUMNS},
    }).set_index("StudyInstanceUID")
    
    ds = Knee25DDataset(
        series_df=series_df,
        labels_df=dummy_labels,
        dicom_root=str(dicom_path),
        planes=c4["data"]["planes"],
        image_size=c4["data"]["image_size"],
        slice_count=c4["data"]["slice_count"],
        is_train=True,
    )
    
    if len(ds) > 0:
        print(f"Dataset: {len(ds)} samples → 2-epoch dry run")
        print(f"{'='*55}")
        
        loader = DataLoader(ds, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
        
        # 测试每个 backbone
        for arch_name, model_cls in [
            ("efficientnetv2_s", EfficientNetV2S25D),
            ("convnext_small", ConvNeXt25D),
            ("swin_tiny", Swin25D),
        ]:
            model = model_cls(
                in_channels=5, num_classes=12, pretrained=False, dropout=0.3,
            ).to(device)
            
            criterion = FocalBCELoss(gamma=2.0, alpha=0.25)
            optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
            scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None
            
            t0 = time.time()
            for epoch in range(2):
                model.train()
                total_loss = 0.0
                for batch in loader:
                    images = batch["image"].to(device)
                    labels = batch["labels"].to(device)
                    
                    with torch.amp.autocast("cuda", enabled=scaler is not None):
                        logits = model(images)
                        loss = criterion(logits, labels)
                    
                    if scaler is not None:
                        scaler.scale(loss).backward()
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()
                    
                    optimizer.zero_grad()
                    total_loss += loss.item()
                
                avg_loss = total_loss / len(loader)
                print(f"  [{arch_name}] epoch {epoch}: loss={avg_loss:.4f}")
            
            elapsed = time.time() - t0
            print(f"  [{arch_name}] 2 epochs: {elapsed:.1f}s OK")
            del model, criterion, optimizer
            torch.cuda.empty_cache() if device == "cuda" else None
        
        print(f"{'='*55}")
        print("Ensemble backbone 干跑完成 — 所有 3 个 backbone 训练循环无报错!")
    else:
        print(f"⚠️  Dataset 为空 — DICOM 不可用")
else:
    print(f"⚠️  跳过 backbone 干跑 — 数据不存在")
    print(f"   series_csv exists: {series_path.exists()}")
    print(f"   dicom_root exists: {dicom_path.exists()}")

---
## 调试结论

Cell 1-2, 4-5, 7-11 不需要 DICOM 即可完整测试。Cell 3, 6, 12 需要 DICOM 数据。

### Phase 3 常见问题速查

| 现象 | 可能原因 | 检查 |
|------|----------|------|
| VolumeDataset 样本数为 0 | 无 Sagittal 系列或 DICOM 缺失 | Cell 3 确认 best_series 选择 |
| volume [0,0,0,0] 全零 | DICOM 读取路径错误 | 检查 dicom_dir 存在且有 .dcm 文件 |
| ResNet3D forward shape error | 通道数不匹配 | Cell 4 验证 in_channels=1 |
| VRAM OOM | batch_size > 1 或未开 grad ckpt | bs=1, grad_ckpt=True, AMP on |
| 梯度为 None | use_grad_checkpoint 与某些层冲突 | Cell 5 对比 on/off 模式 |
| 3D AUC 低于 2.5D | volume 深度/尺寸不合适 | 尝试 depth=64, size=160 |
| Slice 索引范围错误 | volume 切片数 < depth | padding 策略: replicate edges |

### Phase 4 常见问题速查

| 现象 | 可能原因 | 检查 |
|------|----------|------|
| ConvNeXt/Swin 预训练权重加载失败 | timm 版本不匹配 | pip install timm>=0.9.2 |
| Swin forward shape error | img_size 不兼容 Swin window | 用 224 或 384 (被 32 整除) |
| EnsembleModel weight 不更新 | learnable_weights=False | Cell 9 检查 model_weights.grad |
| EnsembleInference load_state_dict 失败 | ckpt key 不含 "model." | 检查 checkpoint 保存格式 |
| ensemble_submissions shape 不一致 | 各 submission index 不同 | 用 index intersection |
| Integrated AUC < 单模型最佳 | 权重不合理 | 基于 val AUC 调权重 |

### 运行命令

```bash
# Phase 3: 3D 训练
python train.py --config configs/phase3_3d.yaml

# Phase 3: 3D 快速验证 (2 epochs)
python train.py --config configs/phase3_3d.yaml --epochs 2

# Phase 4: 单个 backbone 训练 (修改 config model.arch)
python train.py --config configs/phase4_ensemble.yaml  # arch=convnext_small

# Phase 4: 切换 backbone — 修改配置文件中的 model.arch:
#   arch: efficientnetv2_s   → 训练 EfficientNetV2-S
#   arch: convnext_small     → 训练 ConvNeXt-S
#   arch: swin_tiny          → 训练 Swin-T

# Phase 4: Ensemble 合并 inference
python train.py --config configs/phase4_ensemble.yaml --ensemble
```

### Phase 3-4 文件结构

```
├── configs/phase3_3d.yaml          ← 3D ResNet 训练配置 (bs=1, accum=8, grad_ckpt)
├── configs/phase4_ensemble.yaml    ← Ensemble backbone 训练 + 集成推理配置
├── models/resnet3d.py              ← ResNet3DModel (grad_ckpt, 3D pool, extract_features)
├── models/convnext.py              ← ConvNeXt25D (5ch, GAP, extract_features)
├── models/swin.py                  ← Swin25D (5ch, dim-agnostic pool, extract_features)
├── models/ensemble.py              ← EnsembleModel + EnsembleInference + ensemble_submissions
├── datasets/volume_dataset.py      ← VolumeDataset (best Sag series, center crop, replicate pad)
├── train.py                        ← 训练/验证/推理 (phase3/phase4 CLI routing)
└── notebooks/07_phase3_4_debug.ipynb ← 本调试 notebook
```